In [1]:
# 08_run_ablations.ipynb
# Multi-seed ablation of the Q-G-CTGAN two-stage filter (Reviewer #8, comments 3 & 4).
# Reuses the SAME generative base (qgctgan_prepare) per (dataset, seed) and varies only
# the filter configuration, over 5 seeds. Three ablations are produced:
#   (A) quality-score components : full (1/3,1/3,1/3) vs density-only / sep-only / lof-only
#                                  (fixed alpha=0.2, MMD stage ON)
#   (B) threshold               : fixed alpha in {0.1,0.2,0.3} vs adaptive  (full weights, MMD ON)
#   (C) MMD Stage-2             : adaptive alpha, MMD ON vs OFF             (full weights)
# Output: ../results/tables/raw_ablation.csv   ENV: the ctgan env (ctgan + scikit-learn + lightgbm + scipy).
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             average_precision_score, balanced_accuracy_score, confusion_matrix)
import qgctgan_core as Q      # qgctgan_prepare, quality_score, compute_adaptive_alpha,
                             # median_bandwidth, mmd_contribution, TARGET_RATIO
warnings.filterwarnings("ignore")

DATA_DIR   = "../data"
TABLES_DIR = "../results/tables"
os.makedirs(TABLES_DIR, exist_ok=True)
OUT = os.path.join(TABLES_DIR, "raw_ablation.csv")

SEEDS = [42, 43, 44, 45, 46]
DATASETS_SUBSET = None
DATASETS = ["credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
            "yeast_me2", "mammography", "abalone_19", "wine_quality", "ecoli",
            "pageblocks", "protein_homo", "satellite", "churn", "secom",
            "thyroid_sick", "unsw_nb15"]
CLASSIFIERS = ("RF", "LGBM", "MLP")
METRICS = ["AUC", "PR_AUC", "F1", "Precision", "Recall", "G_mean", "Balanced_Acc"]
RAW_COLUMNS = ["dataset", "config", "config_group", "classifier", "seed"] + METRICS

# ---- filter configurations ----
CONFIGS = (
    [dict(config=f"comp_{n}", group="component", w=w, mode="fixed", af=0.2, mmd=True)
     for n, w in [("full", (1/3, 1/3, 1/3)), ("density", (1, 0, 0)),
                  ("sep", (0, 1, 0)), ("lof", (0, 0, 1))]]
    + [dict(config=f"alpha_fixed_{a}", group="alpha", w=(1/3, 1/3, 1/3), mode="fixed", af=a, mmd=True)
       for a in (0.1, 0.2, 0.3)]
    + [dict(config="alpha_adaptive", group="alpha", w=(1/3, 1/3, 1/3), mode="adaptive", af=None, mmd=True)]
    + [dict(config="mmd_on",  group="mmd", w=(1/3, 1/3, 1/3), mode="adaptive", af=None, mmd=True),
       dict(config="mmd_off", group="mmd", w=(1/3, 1/3, 1/3), mode="adaptive", af=None, mmd=False)]
)

# ---- shared helpers (same protocol as the main run) ----
def load_dataset(name):
    df = pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"))
    bc = df.select_dtypes(include="bool").columns
    if len(bc): df[bc] = df[bc].astype("float64")
    return df.drop(columns=["target"]).values.astype(float), df["target"].values.astype(int)

def get_classifier(name, seed):
    if name == "RF":
        return RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=3,
                                      class_weight="balanced", random_state=seed, n_jobs=-1)
    if name == "LGBM":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31,
                              class_weight="balanced", random_state=seed, n_jobs=-1, verbose=-1)
    return MLPClassifier(hidden_layer_sizes=(128, 64), alpha=0.001, max_iter=300, random_state=seed)

def g_mean_score(yt, yp):
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    se = tp/(tp+fn) if (tp+fn) else 0.0; sp = tn/(tn+fp) if (tn+fp) else 0.0
    return float(np.sqrt(se*sp))

def evaluate(m, Xte, yte):
    pr = m.predict_proba(Xte)[:, 1]; pd_ = m.predict(Xte)
    return {"AUC": round(roc_auc_score(yte, pr), 4), "PR_AUC": round(average_precision_score(yte, pr), 4),
            "F1": round(f1_score(yte, pd_, average="macro", zero_division=0), 4),
            "Precision": round(precision_score(yte, pd_, average="macro", zero_division=0), 4),
            "Recall": round(recall_score(yte, pd_, average="macro", zero_division=0), 4),
            "G_mean": round(g_mean_score(yte, pd_), 4),
            "Balanced_Acc": round(balanced_accuracy_score(yte, pd_), 4)}

def seed_everything(s):
    np.random.seed(s)
    try:
        import torch; torch.manual_seed(s)
    except Exception:
        pass

def already_done(csv, cfg):
    d = set()
    if os.path.exists(csv):
        try:
            p = pd.read_csv(csv, usecols=["dataset", "config", "seed"], keep_default_na=False)
            p = p[p["config"] == cfg]
            d = set((str(x), int(y)) for x, y in p[["dataset", "seed"]].drop_duplicates().values)
        except Exception:
            pass
    return d

# ---- configurable Stage-1/Stage-2 filter (same logic as qgctgan_core.filter_by_quality,
#      but the quality score takes the ablation weights w) ----
def filter_configurable(prepared, w, mode, af, mmd_on):
    X_min, X_maj = prepared["X_min"], prepared["X_maj"]
    best_k, gmm, labels = prepared["best_k"], prepared["gmm"], prepared["labels"]
    candidates = prepared["candidates"]
    n_maj, n_min = len(X_maj), len(X_min)
    n_need = max(int(np.round((Q.TARGET_RATIO*(n_maj+n_min)-n_min)/(1-Q.TARGET_RATIO))), 0)
    alpha_vec = (Q.compute_adaptive_alpha(gmm, X_min, labels, best_k) if mode == "adaptive"
                 else np.full(best_k, af))
    mu_min_all = gmm.means_; mu_maj = np.array([X_maj.mean(axis=0)])
    kept = []
    for c in range(best_k):
        Xc, Xr = candidates[c], X_min[labels == c]
        if len(Xc) == 0 or len(Xr) < 2:
            continue
        q = Q.quality_score(Xc, gmm.means_[c], gmm.covariances_[c], mu_min_all, mu_maj, Xr, w=w)
        kept.append(Xc[q >= np.quantile(q, alpha_vec[c])])
    pool = [k for k in kept if len(k) > 0]
    if not pool:
        X_syn = np.empty((0, X_min.shape[1]))
    else:
        P = np.vstack(pool)
        if len(P) <= n_need:
            X_syn = P
        elif not mmd_on:
            X_syn = P[:n_need]
        else:
            gamma = Q.median_bandwidth(X_min, P)
            sc = np.array([Q.mmd_contribution(P[i], X_min, P, gamma) for i in range(len(P))])
            X_syn = P[np.argsort(sc)[:n_need]]
    if len(X_syn):
        Xo = np.vstack([X_maj, X_min, X_syn])
        yo = np.concatenate([np.zeros(n_maj), np.ones(n_min), np.ones(len(X_syn))]).astype(int)
    else:
        Xo = np.vstack([X_maj, X_min]); yo = np.concatenate([np.zeros(n_maj), np.ones(n_min)]).astype(int)
    return Xo, yo

# ---- run: prepare ONCE per (dataset, seed), then evaluate every config ----
datasets = DATASETS_SUBSET or DATASETS
done = {cfg["config"]: already_done(OUT, cfg["config"]) for cfg in CONFIGS}
for seed in SEEDS:
    for ds in datasets:
        if all((ds, seed) in done[cfg["config"]] for cfg in CONFIGS):
            print(f"[skip-done] all configs {ds} seed={seed}"); continue
        try:
            X, y = load_dataset(ds)
        except FileNotFoundError:
            print(f"[missing csv] {ds}"); continue
        try:
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=seed)
            sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
            seed_everything(seed)
            prepared = Q.qgctgan_prepare(Xtr, ytr, random_state=seed)   # shared generative base
            for cfg in CONFIGS:
                if (ds, seed) in done[cfg["config"]]:
                    continue
                Xr, yr = filter_configurable(prepared, cfg["w"], cfg["mode"], cfg["af"], cfg["mmd"])
                rows = []
                for cn in CLASSIFIERS:
                    clf = get_classifier(cn, seed); clf.fit(Xr, yr)
                    rows.append({"dataset": ds, "config": cfg["config"], "config_group": cfg["group"],
                                 "classifier": cn, "seed": seed, **evaluate(clf, Xte, yte)})
                pd.DataFrame(rows, columns=RAW_COLUMNS).to_csv(
                    OUT, mode="a", header=not os.path.exists(OUT), index=False)
                print(f"[ok] {cfg['config']:16s} seed={seed} {ds:16s} "
                      f"AUC(RF/LGBM/MLP)={rows[0]['AUC']}/{rows[1]['AUC']}/{rows[2]['AUC']}")
        except Exception as e:
            print(f"[FAIL-skip] {ds} seed={seed} -> {type(e).__name__}: {e}")
            continue
print("DONE ->", OUT)

  GMM: best_k=9  cluster_sizes=[378, 539, 461, 1040, 335, 181, 478, 609, 624]  [5.3s]
  Target synthetic samples (Delta, 50% balance): 11,710
  Candidate multiplier: 5.0x [within cap]
  Training 9 CTGAN models...
    Cluster 0: n=378  OK
    Cluster 1: n=539  OK
    Cluster 2: n=461  OK
    Cluster 3: n=1040  OK
    Cluster 4: n=335  OK
    Cluster 5: n=181  OK
    Cluster 6: n=478  OK
    Cluster 7: n=609  OK
    Cluster 8: n=624  OK
  Generating candidates (x5.0)...
  Total candidates generated: 23,225  [0.8s]  (target=11,710, sufficient)
[ok] comp_full        seed=42 credit_default   AUC(RF/LGBM/MLP)=0.7628/0.7808/0.7064
[ok] comp_density     seed=42 credit_default   AUC(RF/LGBM/MLP)=0.7634/0.7803/0.6749
[ok] comp_sep         seed=42 credit_default   AUC(RF/LGBM/MLP)=0.7641/0.7801/0.6785
[ok] comp_lof         seed=42 credit_default   AUC(RF/LGBM/MLP)=0.7645/0.7801/0.7186
[ok] alpha_fixed_0.1  seed=42 credit_default   AUC(RF/LGBM/MLP)=0.7646/0.7817/0.6903
[ok] alpha_fixed_0.2  seed=4